# Week 2 Day 2 Lab - Agentic RAG, MCP and Memory

*Arthur's version: the assignment, with a word lookup and a hybrid search in place of Qdrant, a retrieval test that needs no model, notes on each customer, and a researcher sub-agent.*

## Giving the agent a long-term memory

Another 25 claims have arrived at UnderCover, and most of them turn on something you won't find in the claim: when a storm was named, what a device is, or how deep a dive site goes.

We give an agent a long-term memory: Qdrant, a vector database, running as an MCP server. We teach it the facts it's missing, then run the UnderCover agent with and without that memory as experiments in Langfuse.

## 4 parts:

PART 1: The even harder claims

PART 2: A memory over MCP

PART 3: Teaching the agent

PART 4: Experiments in Langfuse

PART 5: The assignment

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">This lab calls GPT-6 Luna through OpenRouter and costs a few cents. If you'd rather not spend anything, there's a free version of this lab in <a href="../../labs/week2day2free.ipynb">week2day2free.ipynb</a>.<br/><br/>
            You'll need your Langfuse keys from Day 1. Teaching the agent takes about 3 minutes, and each experiment takes 3 or 4 minutes to run. For a quicker run, set <code>TINY = True</code> in Part 4 to use 5 claims in place of 25.<br/><br/>
            Please execute each cell and check that you get similar results to mine (the models vary a little from run to run). Some cells have code that isn't about AI, like the charts. It's not important to understand those in detail.
            </span>
        </td>
    </tr>
</table>

In [ ]:
import json
import math
import os
import re
import shutil
import sys
from collections import Counter
from datetime import date
from pathlib import Path
from typing import Literal

import httpx
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import ToolRetryMiddleware
from langchain_core.tools import tool
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_openai import ChatOpenAI
from langfuse import Evaluation, get_client
from langfuse.langchain import CallbackHandler
sys.path.append("../../labs")
from week2day1_payout import Benefit, Line, Payout, work_out_payout

## Connect to OpenRouter and Langfuse

The keys come from the `.env` file in the `undercover` folder, including the Langfuse keys you added on Day 1.

In [ ]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
langfuse = get_client()

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

if langfuse.auth_check():
    print("Connected to Langfuse")
else:
    print("No Langfuse keys found. Add them to the .env file")

## Part 1: The even harder claims

The new claims are in `data/evals/even_harder`, with their policy records and the handlers' decisions.

In [ ]:
EVEN_HARDER = Path("../../data/evals/even_harder")


def load(name):
    """Read one of the JSON files of the even harder claims."""
    return json.loads((EVEN_HARDER / name).read_text(encoding="utf-8"))


claims = load("claims.json")
labels = {decision["claim_id"]: decision for decision in load("decisions.json")}
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = Path("../../data/policy_wording.md").read_text(encoding="utf-8")

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")

In [ ]:
rows = []
for claim in claims:
    label = labels[claim["claim_id"]]
    rows.append({
        "claim_id": claim["claim_id"],
        "summary": claim["summary"],
        "amount_claimed": claim["amount_claimed"],
        "decision": label["decision"],
        "payout": label["payout"],
    })

data = pd.DataFrame(rows)
data

### The knowledge base

The facts the handlers used are in `data/knowledge.json`: news about storms and strikes, a catalogue of devices, and notes on dive sites, rivers and activity operators. Some of them aren't needed by any claim.

In [ ]:
knowledge = json.loads(Path("../../data/knowledge.json").read_text(encoding="utf-8"))

pd.set_option("display.max_colwidth", None)
pd.DataFrame(knowledge)

## Part 2: A memory over MCP

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">MCP</h2>
            <span style="color:#118AB2;">MCP, the Model Context Protocol, is an open standard for connecting agents to tools. An MCP server offers tools, and an MCP client makes them available to an agent.<br/><br/>
            Qdrant's MCP server has two tools: <code>qdrant-store</code> saves a piece of text in a vector database, and <code>qdrant-find</code> searches it by meaning. We run it on our own machine with <code>uvx</code>, and the database lives in a local folder.
            </span>
        </td>
    </tr>
</table>

### One adjustment for Windows

When Python launches another program, it hands that program a place to write its error messages. Inside a Jupyter notebook on Windows, that place is not a real file, and the library that launches MCP servers trips over it. The cell below points the MCP server's error log somewhere safe before we connect. On Mac and Linux it does nothing at all, so run it either way and carry on.

In [ ]:
import sys

if sys.platform == "win32":
    import subprocess
    from functools import partial
    import langchain_mcp_adapters.sessions as mcp_sessions

    mcp_sessions.stdio_client = partial(mcp_sessions.stdio_client, errlog=subprocess.DEVNULL)
    print("Applied the Windows adjustment")
else:
    print("Not Windows, so nothing to do here")

### Connecting to the memory server

Qdrant keeps its database in the folder `qdrant_data`. We delete it first, so the agent starts with an empty memory. The first time you connect, `uvx` downloads the server and a small embedding model, which takes about half a minute.

**Changed:** `--python 3.12` tells `uvx` which Python to run the server with. Without it, `uvx` picks the newest Python on the machine, and on Python 3.14 one of the server's dependencies, `pydantic-core`, has no ready-made package and fails to build. The server then never starts, and the only error is `McpError: Connection closed`.

In [ ]:
MEMORY = Path("qdrant_data").resolve()
shutil.rmtree(MEMORY, ignore_errors=True)

In [ ]:
MEMORY_ENV = {"QDRANT_LOCAL_PATH": str(MEMORY), "COLLECTION_NAME": "memory"}

parameters = {"transport": "stdio", "command": "uvx", "args": ["--python", "3.12", "mcp-server-qdrant@0.8.1"], "env": MEMORY_ENV}

client = MultiServerMCPClient({ "memory": parameters})

memory_tools = await client.get_tools()
for memory_tool in memory_tools:
    print(f"{memory_tool.name}: {memory_tool.description}\n")

Each tool call starts the server, runs the tool and stops the server again. Qdrant's local mode lets only one process open the database at a time, so when the agent searches twice at once, one of the searches fails. `ToolRetryMiddleware` tries a failed tool call again after a short wait.

### An agent with a memory

The memory tools come from the server; the instructions tell the agent how to use them.

The MCP tools only run asynchronously so we need to use async python. In Part 4, Langfuse runs each experiment on its own event loop, and a connection opened on one event loop can't be reused on another, so we give the model an HTTP client that doesn't keep connections open between calls.

In [ ]:
MEMORY_INSTRUCTIONS = """You have a long-term memory.
When you are told a fact, store it in your memory word for word.
When you are asked a question, search your memory and answer from what you find. If your memory has nothing on it, say that you don't know.
"""

http_client = httpx.AsyncClient(limits=httpx.Limits(max_keepalive_connections=0))
luna = ChatOpenAI(model="openai/gpt-6-luna", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, reasoning_effort="low", timeout=120, max_retries=5, http_async_client=http_client)

assistant = create_agent(model=luna, tools=memory_tools, system_prompt=MEMORY_INSTRUCTIONS, middleware=[ToolRetryMiddleware(max_retries=3)])

`ask` starts a new conversation every time, so the agent can't answer from earlier messages. Whatever it knows next time has to come from its memory.

In [ ]:
async def ask(message):
    """Send a message to the assistant in a new conversation and return its reply."""
    result = await assistant.ainvoke({"messages": [{"role": "user", "content": message}]})
    return result["messages"][-1].text


print(await ask("What kind of device is the Halden Arc?"))
print(await ask("When was Tropical Storm Marisol named?"))

## Part 3: Teaching the agent

Now we tell the agent each fact in the knowledge base, one message at a time. This takes about 3 minutes.

In [ ]:
for entry in knowledge:
    await ask(f"Please remember this: {entry['fact']}")
    print(f"Told: {entry['fact'][:80]}...")

We ask the same questions again, each in a new conversation.

In [ ]:
print(await ask("What kind of device is the Halden Arc?"))
print(await ask("When was Tropical Storm Marisol named?"))

A tool is still a function, so we can search the memory ourselves. The search works on meaning, so a question can find a fact that shares none of its words. It returns the 10 closest facts, nearest first.

In [ ]:
search = [memory_tool for memory_tool in memory_tools if memory_tool.name == "qdrant-find"][0]

results = await search.ainvoke({"query": "a gadget for watching films on a wall"})
for line in json.loads(results[0]["text"]):
    print(line)

## Part 4: Experiments in Langfuse

### The UnderCover agent

The instructions, the prompt, `record_decision` and the payout tool are the same as on Day 1.

In [ ]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.

Use your tools to record your decision.
"""

PAYOUT_TOOL_LINE = "Use the payout tool to work out the payout on an approved claim, and record the payout it returns.\n"


def make_user_prompt(claim):
    """The customer's policy record and the claim, as JSON."""
    policy = policies[claim["policy_id"]]
    return f"""# Policy record

{json.dumps(policy, indent=2)}

# Claim

{json.dumps(claim, indent=2)}
"""

In [ ]:
recorded = {}


@tool(parse_docstring=True)
def record_decision(claim_id: str, rationale: str, decision: Literal["approve", "decline", "refer"], payout: float) -> str:
    """Record the decision on a claim in the claims system.

    Args:
        claim_id: The claim's id
        rationale: 2-3 sentences to explain the decision along with the relevant section of the policy
        decision: The decision on the claim
        payout: The amount to pay in US dollars, or 0 for a decline or a referral
    """
    if claim_id in recorded:
        return f"A decision is already recorded on {claim_id}, and decisions are final"
    if decision == "approve" and payout <= 0:
        return "An approval needs a payout above zero"
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"


@tool(parse_docstring=True)
def calculate_payout(policy_id: str, benefit: Benefit, lines: list[Line], paid_by_others: float = 0.0) -> Payout:
    """Work out the payout on an approved claim with UnderCover's payout rules.

    Args:
        policy_id: The customer's policy id
        benefit: The benefit the claim is paid under
        lines: The covered items and expenses, one for each line of the claim, with any excluded ones left out. A pair or a set counts as one item
        paid_by_others: Anything a carrier, hotel or other party has paid or refunded for the same loss

    Returns:
        The calculation, the payout and whether it needs sign-off
    """
    return work_out_payout(policies[policy_id], benefit, lines, paid_by_others)

### Two agents

Both agents run Luna at `low` with the payout tool. The second one also gets `qdrant-find`, so it can search the memory whenever it decides to. This is agentic RAG: the agent chooses when to retrieve and what to search for. Its server runs in read-only mode, so it can search the memory but can't change it.

In [ ]:
read_only_client = MultiServerMCPClient({
    "memory": {"transport": "stdio", "command": "uvx", "args": ["--python", "3.12", "mcp-server-qdrant@0.8.1"], "env": {**MEMORY_ENV, "QDRANT_READ_ONLY": "true"}}
})

memory_search = await read_only_client.get_tools()
print([memory_tool.name for memory_tool in memory_search])

In [ ]:
MEMORY_LINE = "Before you decide, search your memory for any product, event or place in the claim that you need to know more about.\n"

payout_agent = create_agent(model=luna, tools=[record_decision, calculate_payout], system_prompt=f"{INSTRUCTIONS}{PAYOUT_TOOL_LINE}\n{policy_wording}\n")

memory_agent = create_agent(
    model=luna,
    tools=[record_decision, calculate_payout, *memory_search],
    system_prompt=f"{INSTRUCTIONS}{PAYOUT_TOOL_LINE}{MEMORY_LINE}\n{policy_wording}\n",
    middleware=[ToolRetryMiddleware(max_retries=3)],
)

### Scoring

The scores are the ones from Day 1, with one more for retrieval. For the claims that need a fact from the knowledge base, `found_fact` is whether the agent's searches brought that fact back.

In [ ]:
DECISIONS = ["approve", "decline", "refer"]
WRONGFUL_DECLINE = 1500
NEEDLESS_REFERRAL = 50


def is_right(answer, expected):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == expected["decision"] and abs(answer["payout"] - expected["payout"]) <= 1


def mistake_cost(answer, expected):
    """What the agent's answer costs UnderCover, against the handler's decision."""
    if answer["decision"] == expected["decision"]:
        return abs(answer["payout"] - expected["payout"]) if answer["decision"] == "approve" else 0
    if answer["decision"] == "decline":
        return WRONGFUL_DECLINE
    if answer["decision"] == "approve":
        return answer["payout"]
    return NEEDLESS_REFERRAL


def found_fact(answer, expected):
    """For a claim that needs a fact from the knowledge base, whether the agent's searches found it."""
    if not expected["needs"]:
        return None
    return all(fact in answer["retrieved"] for fact in expected["needs"])


def kappa(table):
    """Cohen's kappa for the agent's decisions against the handlers'."""
    observed = (table["agent"] == table["handler"]).mean()
    agent_share = table["agent"].value_counts(normalize=True)
    handler_share = table["handler"].value_counts(normalize=True)
    expected = sum(agent_share.get(decision, 0) * handler_share.get(decision, 0) for decision in DECISIONS)
    return (observed - expected) / (1 - expected)


def show_confusion_matrix(table, title):
    """Draw the confusion matrix, with the handler's decision on each row and the agent's in each column."""
    matrix = pd.crosstab(table["handler"], table["agent"]).reindex(index=DECISIONS, columns=DECISIONS, fill_value=0)
    fig = px.imshow(matrix, text_auto=True, color_continuous_scale="Blues", labels={"x": "Agent", "y": "Handler", "color": "Claims"}, title=title)
    fig.show()

In [ ]:
def score_claim(*, output, expected_output, **kwargs):
    """Item evaluator: whether the agent got the claim right, the cost of its mistake, and whether it found the fact it needed."""
    evaluations = [
        Evaluation(name="right", value=int(is_right(output, expected_output))),
        Evaluation(name="mistake_cost", value=mistake_cost(output, expected_output)),
    ]
    found = found_fact(output, expected_output)
    if found is not None:
        evaluations.append(Evaluation(name="found_fact", value=int(found)))
    return evaluations


def score_table(item_results):
    """One row for each claim in a run: the agent's answer beside the handler's, with the scores."""
    rows = []
    for result in item_results:
        answer, expected = result.output, result.item.expected_output
        rows.append({
            "claim_id": result.item.id,
            "handler": expected["decision"],
            "agent": answer["decision"],
            "handler_payout": expected["payout"],
            "agent_payout": answer["payout"],
            "right": is_right(answer, expected),
            "mistake_cost": mistake_cost(answer, expected),
            "found_fact": found_fact(answer, expected),
            "read_notes": answer["read_notes"],
            "llm_cost": answer["cost"],
            "rationale": answer["rationale"],
        })
    return pd.DataFrame(rows).sort_values("claim_id")


def run_scores(table):
    """The scores for a whole run."""
    agent_approvals = table[table["agent"] == "approve"]
    handler_approvals = table[table["handler"] == "approve"]
    return {
        "decision_accuracy": (table["agent"] == table["handler"]).mean(),
        "accuracy": table["right"].mean(),
        "precision": (agent_approvals["handler"] == "approve").mean(),
        "recall": (handler_approvals["agent"] == "approve").mean(),
        "kappa": kappa(table),
        "cost_of_mistakes": table["mistake_cost"].sum(),
        "straight_through": (table["agent"] != "refer").mean(),
        "found_fact": table["found_fact"].dropna().astype(float).mean(),
        "read_notes": table["read_notes"].mean(),
        "llm_cost": table["llm_cost"].sum(),
    }


def score_run(*, item_results, **kwargs):
    """Run evaluator: the scores for the whole run."""
    scores = run_scores(score_table(item_results))
    return [Evaluation(name=name, value=float(value)) for name, value in scores.items()]


def average_scores(table):
    """The scores for each run of an experiment, averaged over the runs."""
    return pd.DataFrame([run_scores(run_table) for _, run_table in table.groupby("run")]).mean()

### The dataset

We upload the even harder claims as a Langfuse dataset, with the handler's decision and the facts each claim needs as the expected output.

In [ ]:
DATASET = "even-harder-claims"

langfuse.create_dataset(name=DATASET, description="UnderCover's even harder claims with the handlers' decisions")

for claim in claims:
    label = labels[claim["claim_id"]]
    langfuse.create_dataset_item(
        dataset_name=DATASET,
        id=claim["claim_id"],
        input=claim,
        expected_output={"decision": label["decision"], "payout": label["payout"], "reason": label["reason"], "needs": label["needs"]},
        metadata={"tiny": label["tiny"]},
    )

As on Day 1, `TINY = True` keeps 5 of the claims, and `N_TIMES` is the number of times each experiment runs.

In [ ]:
TINY = False
N_TIMES = 3

dataset = langfuse.get_dataset(DATASET)
items = dataset.items
if TINY:
    items = [item for item in items if item.metadata["tiny"]]

print(f"Running {len(items)} of the {len(dataset.items)} claims")

### The task

`decide` uses `ainvoke`, since the memory tools only run asynchronously, and Langfuse runs async tasks too. Along with the decision, `decide` keeps what the agent's memory searches returned.

**Changed:** the assignment's agents search with tools of other names, and in the stretch a sub-agent does the searching. So `decide` keeps the results of every tool in `SEARCH_TOOLS`, adds what the sub-agent found and what it cost from `research`, and records whether the customer's notes were read.

In [ ]:
langfuse_handler = CallbackHandler()

SEARCH_TOOLS = {"qdrant-find", "search_knowledge", "search_hybrid"}

# What the researcher sub-agent did for the claim being decided. The experiments run one claim at a time.
research = {"retrieved": [], "tools": set(), "cost": 0.0}


def model_cost(messages):
    """The cost of the model calls in a conversation."""
    return sum(message.response_metadata["token_usage"]["cost"] for message in messages if message.type == "ai")


async def decide(agent, claim):
    """Run an agent on one claim and return its decision, the cost of its model calls and what its memory searches returned."""
    research.update(retrieved=[], tools=set(), cost=0.0)
    result = await agent.ainvoke(
        {"messages": [{"role": "user", "content": make_user_prompt(claim)}]},
        config={"callbacks": [langfuse_handler]},
    )
    answer = recorded.get(claim["claim_id"], {"decision": "refer", "payout": 0.0, "rationale": "Nothing recorded"})
    tool_messages = [message for message in result["messages"] if message.type == "tool"]
    cost = model_cost(result["messages"]) + research["cost"]
    retrieved = "\n".join([message.text for message in tool_messages if message.name in SEARCH_TOOLS] + research["retrieved"])
    read_notes = "read_customer_notes" in {message.name for message in tool_messages} | research["tools"]
    print(f"{claim['claim_id']}: {answer['decision']} ${answer['payout']:,.2f}")
    return {**answer, "cost": cost, "retrieved": retrieved, "read_notes": read_notes}


async def payout_task(*, item, **kwargs):
    """Run the agent with the payout tool on one claim from the dataset."""
    return await decide(payout_agent, item.input)


async def memory_task(*, item, **kwargs):
    """Run the agent with the payout tool and memory on one claim from the dataset."""
    return await decide(memory_agent, item.input)


### Experiment 1: the payout tool

This is the agent from Day 1 with the payout tool. On Day 1 we turned Luna's reasoning off, and today both agents run with Luna at `low`. Each run takes about 3 minutes.

In [ ]:
payout_tables = []
for run in range(1, N_TIMES + 1):
    recorded.clear()
    result = langfuse.run_experiment(
        name="Payout tool",
        description="The agent with the payout tool",
        data=items,
        task=payout_task,
        evaluators=[score_claim],
        run_evaluators=[score_run],
        max_concurrency=1,
    )
    print(result.format())
    payout_tables.append(score_table(result.item_results).assign(run=run))

In [ ]:
payout_table = pd.concat(payout_tables, ignore_index=True)
show_confusion_matrix(payout_table, "Payout tool")

### Experiment 2: the payout tool and memory

The same agent, now able to search its memory. Each run takes about 4 minutes.

In [ ]:
memory_tables = []
for run in range(1, N_TIMES + 1):
    recorded.clear()
    result = langfuse.run_experiment(
        name="Payout tool and memory",
        description="The agent with the payout tool and a memory of the knowledge base",
        data=items,
        task=memory_task,
        evaluators=[score_claim],
        run_evaluators=[score_run],
        max_concurrency=1,
    )
    print(result.format())
    memory_tables.append(score_table(result.item_results).assign(run=run))

In [ ]:
memory_table = pd.concat(memory_tables, ignore_index=True)
show_confusion_matrix(memory_table, "Payout tool and memory")

Open a trace from this experiment in Langfuse to see what the agent searched for and what came back.

### Comparing the experiments

In [ ]:
pd.DataFrame([average_scores(payout_table), average_scores(memory_table)], index=["Payout tool", "Payout tool and memory"]).round(3)

In Langfuse, open **Datasets** and then `even-harder-claims`, and select the runs to compare them claim by claim.

## Part 5: The assignment

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">Challenge: replace Qdrant with a simple word lookup. Write a tool that searches the facts in <code>data/knowledge.json</code> for the words in its query, give it to the memory agent in place of <code>qdrant-find</code>, and run Experiment 2 again. <code>decide</code> picks out the search results by the tool's name, so change <code>qdrant-find</code> there to the name of your tool. How do the scores compare with searching by meaning?<br/><br/>
            Then give the agent a memory about each customer: a Markdown file for each one, named by the claim's <code>customer_id</code>, with a tool to read a customer's notes and a tool to add to them. Each customer has only one claim in this set, so write a few notes yourself to start with. With <code>N_TIMES</code> above 1, what should happen to the notes between runs?<br/><br/>
            Stretch challenge: move the retrieval into a sub-agent that has <code>qdrant-find</code> and the customer memory tools, and let the UnderCover agent call the sub-agent as a tool.<br/><br/>
            I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>

A helper for the experiments below. It runs an experiment `N_TIMES`, with a fresh set of recorded decisions each time, and calls `before_run` first when there's something else to reset.

In [ ]:
def run_experiment(name, description, task, before_run=None):
    """Run an experiment N_TIMES and return the score table of every run."""
    tables = []
    for run in range(1, N_TIMES + 1):
        recorded.clear()
        if before_run:
            before_run()
        result = langfuse.run_experiment(
            name=name,
            description=description,
            data=items,
            task=task,
            evaluators=[score_claim],
            run_evaluators=[score_run],
            max_concurrency=1,
        )
        print(result.format())
        tables.append(score_table(result.item_results).assign(run=run))
    return pd.concat(tables, ignore_index=True)

### A word lookup

`search_knowledge` searches the facts in `data/knowledge.json` for the words in its query. It lowercases the words, drops common words such as "the", and takes the "s" off plurals, so "strikes" finds "strike". Each fact scores the words it shares with the query, and a rare word like "Marisol" counts for more than a common one like "storm". This weighting is called inverse document frequency. The tool returns the 10 best facts in full, in the same form as `qdrant-find`.

In [ ]:
STOP_WORDS = {"a", "an", "and", "are", "as", "at", "be", "by", "for", "from", "has", "have", "in", "is", "it", "its", "of", "on", "or", "that", "the", "their", "to", "was", "were", "with", "my", "our", "we", "i", "when", "what", "which"}


def words(text):
    """The words of a text that count for a search: lowercased, without common words, and without the s of a plural."""
    found = []
    for word in re.findall(r"[a-z0-9]+(?:'[a-z]+)?", text.lower()):
        word = word.removesuffix("'s")
        if word in STOP_WORDS:
            continue
        if len(word) > 3 and word.endswith("s") and not word.endswith("ss"):
            word = word[:-1]
        found.append(word)
    return found


fact_words = [set(words(entry["fact"])) for entry in knowledge]
document_frequency = Counter(word for fact in fact_words for word in fact)
weight = {word: math.log(len(knowledge) / count) + 1 for word, count in document_frequency.items()}


def word_ranking(query, limit=10):
    """The facts that share words with the query, best first, weighting rare words more."""
    query_words = set(words(query))
    scores = [(sum(weight[word] for word in query_words & fact), index) for index, fact in enumerate(fact_words)]
    return [knowledge[index]["fact"] for score, index in sorted(scores, reverse=True) if score > 0][:limit]


@tool(parse_docstring=True)
def search_knowledge(query: str) -> str:
    """Search UnderCover's knowledge base for facts that share words with the query: news about storms and strikes, devices, dive sites, rivers and activity operators.

    Args:
        query: The words to look for, such as the name of a product, event, place or company in the claim
    """
    return json.dumps(word_ranking(query) or ["No facts share any words with the query"], indent=2)

It finds a fact by its name, and "Hurricane Marisol" finds "Tropical Storm Marisol" through the rare word "Marisol". But a description that shares none of the fact's words fails: for the question from Part 3, which Qdrant answered, the word lookup's best match is the Sentinel Wall dive site, because both mention a "wall".

In [ ]:
for query in ["Halden Arc", "Hurricane Marisol", "a gadget for watching films on a wall"]:
    print(f"{query}: {word_ranking(query)[:1]}")

### Extra: a hybrid search

A hybrid search runs both searches and merges their rankings with reciprocal rank fusion. Each fact scores `1 / (60 + rank)` in each list it appears in, so a fact near the top of either list rises, and one near the top of both rises furthest. `search_hybrid` uses the read-only Qdrant server.

Qdrant returns the facts as the agent stored them, which may differ slightly from `knowledge.json`, so `canonical` matches each result to the fact it shares the most words with before the lists are merged.

In [ ]:
qdrant_find = memory_search[0]


def qdrant_entries(results):
    """The text of each entry that qdrant-find returned."""
    try:
        lines = json.loads(results[0]["text"])
    except (json.JSONDecodeError, KeyError, IndexError):
        return []
    return [match.group(1) for line in lines if (match := re.search(r"<content>(.*?)</content>", line, re.S))]


def canonical(entry):
    """The fact in the knowledge base that a stored memory is a copy of, or the memory itself if none is close."""
    entry_words = set(words(entry))
    overlap, index = max((len(entry_words & fact) / len(entry_words | fact), index) for index, fact in enumerate(fact_words))
    return knowledge[index]["fact"] if overlap >= 0.6 else entry


async def semantic_ranking(query):
    """The facts Qdrant finds closest in meaning to the query, nearest first."""
    return [canonical(entry) for entry in qdrant_entries(await qdrant_find.ainvoke({"query": query}))]


def fuse(*rankings, limit=10):
    """Merge rankings with reciprocal rank fusion."""
    scores = Counter()
    for ranking in rankings:
        for rank, fact in enumerate(ranking, start=1):
            scores[fact] += 1 / (60 + rank)
    return [fact for fact, _ in scores.most_common(limit)]


@tool(parse_docstring=True)
async def search_hybrid(query: str) -> str:
    """Search UnderCover's memory both by meaning and by the words in the query: news about storms and strikes, devices, dive sites, rivers and activity operators.

    Args:
        query: What to look for, such as a product, event, place or company in the claim
    """
    found = fuse(word_ranking(query, limit=20), await semantic_ranking(query))
    return json.dumps(found or ["Nothing found"], indent=2)


print(json.loads(await search_hybrid.ainvoke({"query": "a gadget for watching films on a wall"}))[:2])

### Extra: a retrieval test without the model

Before running agents, we can test the searches alone. For each of the claims that needs a fact, we search with the claim's summary, and then with its whole story, and see where the fact lands in the results. `hit@1` is the share of claims where it comes first, and `hit@10` where it's anywhere in the 10 results. This costs nothing and takes a couple of minutes, since each Qdrant search starts its server.

It separates two kinds of failure: a search that can't find the fact, and an agent that searches for the wrong thing.

In [ ]:
def rank_of(needs, ranking):
    """The position of the last needed fact in a ranking, or None if one is missing from it."""
    ranks = [next((rank for rank, fact in enumerate(ranking, start=1) if need in fact), None) for need in needs]
    return None if None in ranks else max(ranks)


retrieval_rows = []
for claim in claims:
    needs = labels[claim["claim_id"]]["needs"]
    if not needs:
        continue
    for field in ["summary", "story"]:
        query = claim[field]
        word, semantic = word_ranking(query), await semantic_ranking(query)
        rankings = {"word": word, "semantic": semantic, "hybrid": fuse(word_ranking(query, limit=20), semantic)}
        for method, ranking in rankings.items():
            retrieval_rows.append({"claim_id": claim["claim_id"], "query": field, "method": method, "needs": needs[0], "rank": rank_of(needs, ranking)})

retrieval = pd.DataFrame(retrieval_rows)
print(f"{retrieval['claim_id'].nunique()} claims need a fact")

In [ ]:
def hits(group):
    """The share of claims with the fact at the top, in the top 3 and in the top 10."""
    return pd.Series({f"hit@{k}": (group["rank"].notna() & (group["rank"] <= k)).mean() for k in (1, 3, 10)})


retrieval.groupby(["query", "method"])[["rank"]].apply(hits).round(2)

The claims where the methods disagree, with the rank of the fact for each one (blank when it isn't in the 10 results).

In [ ]:
ranks = retrieval.pivot(index=["claim_id", "needs", "query"], columns="method", values="rank")
ranks[ranks.isna().any(axis=1) | (ranks.nunique(axis=1) > 1)]

### Experiment 3: a word lookup in place of Qdrant

The memory agent from Experiment 2, with `search_knowledge` in place of `qdrant-find`.

In [ ]:
SEARCH_LINE = "Before you decide, search the knowledge base for any product, event or place in the claim that you need to know more about.\n"

word_agent = create_agent(
    model=luna,
    tools=[record_decision, calculate_payout, search_knowledge],
    system_prompt=f"{INSTRUCTIONS}{PAYOUT_TOOL_LINE}{SEARCH_LINE}\n{policy_wording}\n",
)


async def word_task(*, item, **kwargs):
    """Run the agent with the word lookup on one claim from the dataset."""
    return await decide(word_agent, item.input)


word_table = run_experiment("Word lookup", "The agent with the payout tool and a word lookup of the knowledge base", word_task)
show_confusion_matrix(word_table, "Word lookup")

### Experiment 4: the hybrid search

In [ ]:
hybrid_agent = create_agent(
    model=luna,
    tools=[record_decision, calculate_payout, search_hybrid],
    system_prompt=f"{INSTRUCTIONS}{PAYOUT_TOOL_LINE}{MEMORY_LINE}\n{policy_wording}\n",
    middleware=[ToolRetryMiddleware(max_retries=3)],
)


async def hybrid_task(*, item, **kwargs):
    """Run the agent with the hybrid search on one claim from the dataset."""
    return await decide(hybrid_agent, item.input)


hybrid_table = run_experiment("Hybrid search", "The agent with the payout tool and a hybrid search of its memory", hybrid_task)
show_confusion_matrix(hybrid_table, "Hybrid search")

### A memory about each customer

Each customer gets a Markdown file in `customer_notes`, named by their `customer_id`. Every customer has only one claim in this set, so I wrote a few notes to start with, in `SEED_NOTES`. They are written so they don't change the handlers' decisions:

- **Two notes support a referral:** CUS-3324 says the police report is on its way, and CUS-3325 that the clinic is sending the itemized bill.
- **One note confirms the wording:** CUS-3316 asked before the trip whether rafting is covered, and was told about the grade 3 rule.
- **Three are only background:** a past claim paid without trouble (CUS-3301), a past referral on a different trip (CUS-3307), and a contact preference (CUS-3322). They check that the agent doesn't let an old claim decide a new one.

`read_customer_notes` reads a customer's file, and `add_customer_note` adds a dated line to the end. Notes are only ever added, never rewritten, like the decisions in `record_decision`. Both tools check the id, so an agent can't read or write any other file.

In [ ]:
NOTES = Path("customer_notes").resolve()

SEED_NOTES = {
    "CUS-3301": ["2025-09-03: Baggage delay claim CLM-0920 paid in full; all receipts sent first time."],
    "CUS-3307": ["2025-11-18: Claim CLM-0871 for a camera lost in Rome was referred for a missing police report, then paid when it arrived. That was a different trip."],
    "CUS-3316": ["2026-06-30: Phoned before the trip to ask whether rafting is covered. Told that white-water rafting above grade 3 is an adventure sport and needs the Adventure Sports add-on."],
    "CUS-3322": ["2026-03-12: Prefers email to phone calls."],
    "CUS-3324": ["2026-07-25: Phoned about the Amsterdam theft. The police report has not arrived yet; she will send it as soon as it does."],
    "CUS-3325": ["2026-08-12: Phoned to say he has asked Oia Medical Center for an itemized bill, which the clinic will send."],
}


def reset_notes():
    """Start the customer notes again from the seed notes."""
    shutil.rmtree(NOTES, ignore_errors=True)
    NOTES.mkdir()
    for customer_id, notes in SEED_NOTES.items():
        (NOTES / f"{customer_id}.md").write_text(f"# Notes on {customer_id}\n\n" + "".join(f"- {note}\n" for note in notes), encoding="utf-8")


def notes_file(customer_id):
    """The notes file for a customer, or None if the id isn't a customer id."""
    return NOTES / f"{customer_id}.md" if re.fullmatch(r"CUS-\d{4}", customer_id) else None


@tool(parse_docstring=True)
def read_customer_notes(customer_id: str) -> str:
    """Read UnderCover's notes on a customer, from earlier claims and contact with them.

    Args:
        customer_id: The customer's id, from the claim
    """
    path = notes_file(customer_id)
    if path is None:
        return f"{customer_id} is not a customer id"
    return path.read_text(encoding="utf-8") if path.exists() else f"No notes on {customer_id} yet"


@tool(parse_docstring=True)
def add_customer_note(customer_id: str, note: str) -> str:
    """Add a note to a customer's record. Notes are kept, never changed.

    Args:
        customer_id: The customer's id, from the claim
        note: One line on what happened, for whoever handles this customer next. The tool adds the date
    """
    path = notes_file(customer_id)
    if path is None:
        return f"{customer_id} is not a customer id"
    if not path.exists():
        path.write_text(f"# Notes on {customer_id}\n\n", encoding="utf-8")
    with path.open("a", encoding="utf-8") as notes:
        notes.write(f"- {date.today()}: {note.strip()}\n")
    return f"Added a note to {customer_id}"


reset_notes()
print(read_customer_notes.invoke({"customer_id": "CUS-3324"}))

### Between runs with `N_TIMES` above 1

The notes go back to the seed notes before each run. The agent adds a note on every claim it decides, so without a reset, run 2 would read what run 1 wrote about the same claim, often including run 1's decision. The runs would no longer be independent, and run 2 would partly copy run 1, so the average over runs would make the agent look steadier than it is.

In production the notes are meant to build up, since that is what makes them a memory. Testing that is a different experiment: run claims from the same customers in date order, keep the notes from one claim to the next, and check whether the notes help or whether mistakes carry over.

### Experiment 5: Qdrant and customer notes

In [ ]:
NOTES_LINE = "Read the customer's notes before you decide, using the claim's customer_id. They are background from earlier contact, and the claim's own documents come first. After you record your decision, add a one-line note about this claim to the customer's notes.\n"

notes_agent = create_agent(
    model=luna,
    tools=[record_decision, calculate_payout, *memory_search, read_customer_notes, add_customer_note],
    system_prompt=f"{INSTRUCTIONS}{PAYOUT_TOOL_LINE}{MEMORY_LINE}{NOTES_LINE}\n{policy_wording}\n",
    middleware=[ToolRetryMiddleware(max_retries=3)],
)


async def notes_task(*, item, **kwargs):
    """Run the agent with memory and customer notes on one claim from the dataset."""
    return await decide(notes_agent, item.input)


notes_table = run_experiment("Memory and customer notes", "The agent with the payout tool, Qdrant and notes on each customer", notes_task, before_run=reset_notes)
show_confusion_matrix(notes_table, "Memory and customer notes")

Here are the notes after the last run on the two customers whose claims wait for a document, and on one who had no seed notes.

In [ ]:
for customer_id in ["CUS-3324", "CUS-3325", "CUS-3302"] if not TINY else [item.input["customer_id"] for item in items[:3]]:
    print(read_customer_notes.invoke({"customer_id": customer_id}))

### Stretch: a researcher sub-agent

The researcher has `qdrant-find` and `read_customer_notes`. Given a claim, it searches for what a handler needs to know and replies with a briefing. The UnderCover agent calls it through the `research_claim` tool, and keeps `add_customer_note` for itself, so only the agent that decides writes to the notes.

The searches now happen inside the researcher, out of sight of `decide`, so `research_claim` records what they returned, which tools ran and what the researcher's model calls cost, in the `research` dictionary. `found_fact` and the cost of each claim then still count the researcher's work. The researcher runs inside the agent's tool call, so its trace sits under the agent's in Langfuse.

In [ ]:
RESEARCH_INSTRUCTIONS = """You research claims for the claims handlers at UnderCover Travel Insurance.
You are given a claim. Search the memory for each product, event, place and company in it that a handler might need to know more about, and read the customer's notes using the claim's customer_id.
Reply with a short briefing: each fact you found, quoted word for word, then the customer's notes. Say so when the memory has nothing on something. Don't decide the claim.
"""

researcher = create_agent(
    model=luna,
    tools=[*memory_search, read_customer_notes],
    system_prompt=RESEARCH_INSTRUCTIONS,
    middleware=[ToolRetryMiddleware(max_retries=3)],
)

claims_by_id = {claim["claim_id"]: claim for claim in claims}


@tool(parse_docstring=True)
async def research_claim(claim_id: str) -> str:
    """Ask the researcher what you need to know to decide a claim: facts from the memory about the products, events and places in it, and the customer's notes.

    Args:
        claim_id: The claim's id
    """
    claim = claims_by_id.get(claim_id)
    if claim is None:
        return f"There is no claim {claim_id}"
    result = await researcher.ainvoke({"messages": [{"role": "user", "content": json.dumps(claim, indent=2)}]})
    tool_messages = [message for message in result["messages"] if message.type == "tool"]
    research["retrieved"] += [message.text for message in tool_messages if message.name in SEARCH_TOOLS]
    research["tools"] |= {message.name for message in tool_messages}
    research["cost"] += model_cost(result["messages"])
    return result["messages"][-1].text


RESEARCH_LINE = "Before you decide, call research_claim with the claim's id. It returns the facts you need about the products, events and places in the claim, and the customer's notes, which are background and come after the claim's own documents. After you record your decision, add a one-line note about this claim to the customer's notes.\n"

lead_agent = create_agent(
    model=luna,
    tools=[record_decision, calculate_payout, research_claim, add_customer_note],
    system_prompt=f"{INSTRUCTIONS}{PAYOUT_TOOL_LINE}{RESEARCH_LINE}\n{policy_wording}\n",
)


async def researcher_task(*, item, **kwargs):
    """Run the agent with the researcher sub-agent on one claim from the dataset."""
    return await decide(lead_agent, item.input)


researcher_table = run_experiment("Researcher sub-agent", "The agent with the payout tool and a researcher sub-agent with Qdrant and customer notes", researcher_task, before_run=reset_notes)
show_confusion_matrix(researcher_table, "Researcher sub-agent")

### Comparing all six experiments

`found_fact` is the share of claims needing a fact where the searches brought it back, and `read_notes` the share of claims where the customer's notes were read.

In [ ]:
EXPERIMENTS = {
    "Payout tool": payout_table,
    "Qdrant": memory_table,
    "Word lookup": word_table,
    "Hybrid search": hybrid_table,
    "Qdrant + notes": notes_table,
    "Researcher sub-agent": researcher_table,
}

comparison = pd.DataFrame([average_scores(table) for table in EXPERIMENTS.values()], index=list(EXPERIMENTS))
comparison.round(3)

How often each search found the fact each claim needed, over the runs. The claims where the methods differ are the ones to open in Langfuse.

In [ ]:
found = pd.DataFrame({
    name: table.dropna(subset=["found_fact"]).assign(found_fact=lambda t: t["found_fact"].astype(float)).groupby("claim_id")["found_fact"].mean()
    for name, table in EXPERIMENTS.items()
    if name != "Payout tool"
})
found.insert(0, "needs", [labels[claim_id]["needs"][0] for claim_id in found.index])
found.round(2)

The claims each experiment got wrong in at least one run, with how many runs it got right.

In [ ]:
right = pd.DataFrame({name: table.groupby("claim_id")["right"].mean() for name, table in EXPERIMENTS.items()})
right.insert(0, "handler", [labels[claim_id]["decision"] for claim_id in right.index])
right[(right.drop(columns="handler") < 1).any(axis=1)].round(2)

### My findings

From 3 runs of each experiment on all 25 claims, with GPT-6 Luna at `low` through OpenRouter, on October 24, 2026. The table shows the average for each run.

| | Accuracy | Decision accuracy | Kappa | Cost of mistakes | `found_fact` | LLM cost |
|---|---|---|---|---|---|---|
| Payout tool, no memory | 57% | 63% | 0.33 | $2,469 | 0% | $0.009 |
| Qdrant | 96% | 97% | 0.95 | $67 | 100% | $0.013 |
| Word lookup | 92% | 97% | 0.95 | $614 | 100% | $0.011 |
| Hybrid search | 93% | 95% | 0.89 | $1,046 | 100% | $0.012 |
| Qdrant + customer notes | **97%** | 97% | 0.95 | **$33** | 100% | $0.016 |
| Researcher sub-agent | 96% | 96% | 0.92 | $50 | 100% | $0.030 |

**How does the word lookup compare with searching by meaning?** It found the fact on every claim in every run, just as Qdrant did. The agent doesn't search with the claim's wording: it picks the names out of the items and receipts and searches for those, such as "Corvane Lumen X2 watch electronics wearable smartwatch". The knowledge base files each fact under the same names, so a word lookup is enough.

Its higher cost of mistakes comes from two things, and neither is the search:
- In one run it declined CLM-1304, a $1,500 wrongful decline, with the Meridian press release in front of it.
- In all 3 runs it left an $85 wallet out of CLM-1308, because the wallet had no receipt. The handler paid it.

The hybrid search's $1,046 is the same story: two wrongful declines, each in one run, with the fact found both times. Once every search finds the fact, the differences between the memory experiments come from the model's judgment, and 3 runs of 25 claims can't separate them.

**The retrieval test shows where the word lookup would break.** Searching with the claim's summary, the word lookup found the fact in its 10 results for 74% of claims, against 89% for Qdrant and the hybrid search. It misses devices that the summary only describes, such as "a reading device". The hybrid search ranked the fact first most often (74%). The agents did better than any of these because they write their own queries. A word lookup suits a knowledge base where facts are filed under the names customers use. It would fail on synonyms, descriptions and spelling mistakes, which is where searching by meaning earns its place.

**Customer notes:** the agent read the notes on every claim and wrote a sensible note on each one. It had the lowest cost of mistakes, but its only two mistakes were needless referrals, so the notes didn't measurably change any decisions. The two claims waiting for a document (CLM-1324 and CLM-1325) were referred correctly with and without notes. The notes didn't mislead it either: the old referral on CUS-3307's different trip didn't lead to a referral this time.

One problem: the agent writes its own date at the start of each note, so the tool's date and the agent's end up side by side ("2026-10-24: 2026-08-01: Referred baggage theft claim..."). The agent's dates are made up, and this still happened after the tool's description said the tool adds the date. The fix belongs in the tool: strip a leading date, or keep the date in a separate field.

**With `N_TIMES` above 1,** the notes go back to the seed notes before each run, so that run 2 can't read run 1's note on the same claim.

**The researcher sub-agent** was as accurate as Qdrant, and its three mistakes were all cautious referrals. But it costs more than twice as much per claim, because every claim runs a second agent loop. It keeps the deciding agent's context small and makes retrieval reusable, which would matter with more sources to search. At this size it isn't worth the cost.

**My choice:** Qdrant with customer notes. Searching by meaning costs little more than a word lookup and holds up when a claim describes something in its own words. The notes cost about $0.003 per run and set the agent up to remember customers once they have more than one claim.

**Also changed from the lab:** `--python 3.12` in both `uvx` commands. Without it, `uvx` picks Python 3.14 when it's installed, the server's `pydantic-core` dependency can't be built, and the cell fails with `McpError: Connection closed`.